[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jman4162/sensortwin-transformer-agent/blob/master/notebooks/05_calibration_colab.ipynb)

# Calibrate the transformer (temperature scaling)

The transformer is the most accurate model at `colab_standard` but the least calibrated (ECE ~0.095). This trains it once at 20k, fits a single temperature on the validation logits (`evaluation/calibration.py`), and re-measures ECE + reliability. Temperature scaling does not change argmax, so macro-F1 is unchanged — the goal is 'most accurate **and** well-calibrated after a 1-parameter fit'. **Runtime → GPU**; ~10 min on a T4.

In [ ]:
# Opened from the Colab badge? Only the notebook is present — clone the public repo, then install.
import os

if not os.path.exists("sensortwin"):
    !git clone https://github.com/jman4162/sensortwin-transformer-agent.git
    %cd sensortwin-transformer-agent
%pip install -q -e ".[ml]"

In [ ]:
import torch

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import numpy as np
from sensortwin.data.dataset import SensorArrayDataset
from sensortwin.data.splits import make_split
from sensortwin.data.transforms import ChannelStandardizer
from sensortwin.simulation import GenConfig, generate_dataset
from sensortwin.simulation.events import EVENT_CLASSES
from sensortwin.utils.config import load_yaml
from sensortwin.utils.seeds import set_torch_seed
from sensortwin.models.transformer import SensorPatchTST
from sensortwin.training.augment import build_augment
from sensortwin.training.loop import class_weights, train_model

set_torch_seed(0)
X, y, _ = generate_dataset(GenConfig(n_samples=20_000, T=512, seed=0, normalize=False))
sp = make_split("random", y, {}, seed=0)
tr, va, te = sp["train"], sp["val"], sp["test"]
std = ChannelStandardizer().fit(X[tr])
val_ds = SensorArrayDataset(std.transform(X[va]), y[va]).as_torch()
test_ds = SensorArrayDataset(std.transform(X[te]), y[te]).as_torch()

cfg = load_yaml("configs/models/sensorpatchtst.yaml")
model = SensorPatchTST(**cfg["model"])
tcfg = cfg["train"]
keys = ("optimizer","lr","weight_decay","label_smoothing","scheduler","warmup_epochs","batch_size","patience")
kw = {k: tcfg[k] for k in keys if k in tcfg}
aug = build_augment(tcfg.get("augment"))
if aug is not None:
    kw["augment"] = aug
train_ds = SensorArrayDataset(std.transform(X[tr]), y[tr]).as_torch()
model, _ = train_model(model, train_ds, val_ds, epochs=15,
                       weight=class_weights(y[tr], len(EVENT_CLASSES)), **kw)
print("trained.")

In [ ]:
from sensortwin.training.loop import predict_logits
from sensortwin.evaluation.calibration import (apply_temperature, expected_calibration_error,
                                               fit_temperature, reliability_curve)
from sensortwin.evaluation.plots import plot_reliability
from IPython.display import Image, display

val_logits = predict_logits(model, val_ds)
test_logits = predict_logits(model, test_ds)
T = fit_temperature(val_logits, y[va])
before = apply_temperature(test_logits, 1.0)   # plain softmax
after = apply_temperature(test_logits, T)
ece_b = expected_calibration_error(y[te], before)
ece_a = expected_calibration_error(y[te], after)
acc_same = (before.argmax(1) == after.argmax(1)).mean()
print(f"temperature T = {T:.3f}")
print(f"ECE  {ece_b:.3f} -> {ece_a:.3f}   (argmax unchanged on {acc_same*100:.1f}% of test -> macro-F1 identical)")
plot_reliability(reliability_curve(y[te], before), "reliability_before.png", title=f"before (ECE {ece_b:.3f})")
plot_reliability(reliability_curve(y[te], after), "reliability_after.png", title=f"after T={T:.2f} (ECE {ece_a:.3f})")
display(Image("reliability_before.png"), Image("reliability_after.png"))

Paste the `ECE before -> after` line back into the chat and I'll fold it into the model card's calibration section.